# 15 · Neural Networks (MLP) & Backpropagation
> **Part:** Part 2 · Neural networks & deep learning  
> **Family:** `Supervised · Deep learning (multi-layer perceptron)`

## Definition
A multi-layer perceptron stacks layers of neurons. Each layer computes a linear map followed by a non-linear activation, so the
network can bend space and learn *any* smooth decision boundary (universal approximation). **Backpropagation** is the chain
rule applied efficiently from the loss back to every weight, and gradient descent then updates them all.

## Intuition
The first layers learn to re-describe the inputs (bend and stretch the plane) so that the last layer, a plain linear classifier,
can separate the classes. Without non-linear activations, stacked layers collapse into one linear layer.


## The Math

```text
a1 = tanh(X W1 + b1)      a2 = tanh(a1 W2 + b2)      z = a2 W3 + b3      p = softmax(z)
    L = −1/n Σ log p[i, yᵢ]
    Backward:  dz  = (p − onehot(y)) / n
               dW3 = a2ᵀ dz            da2 = dz W3ᵀ      dz2 = da2 ⊙ (1 − a2²)
               dW2 = a1ᵀ dz2           da1 = dz2 W2ᵀ     dz1 = da1 ⊙ (1 − a1²)
               dW1 = Xᵀ dz1            (biases: column sums)        then  W ← W − η·dW (here with Adam)
    This file writes every line of that by hand and verifies it against numerical gradients.
```


In [ ]:
# Setup, Imports & Styling
%matplotlib inline
import os
import pathlib
import sys
from IPython.display import Image, display

# Ensure parent folder is in path to import mlatlas
parent_dir = pathlib.Path.cwd().parent
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from mlatlas import data, viz, cli

# Ensure interactive inline mode and configure Seaborn theme
plt.ion()
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"figure.dpi": 100, "figure.facecolor": "white"})


### Step 1: Input Representation & Target Geometry
We visualize the synthetic input distributions and representation structure for **Neural Networks (MLP) & Backpropagation**.


In [ ]:
# Explore dataset representation
print("Neural architecture analysis for Neural Networks (MLP) & Backpropagation")
sns.set_theme(style="whitegrid", palette="muted")


### Step 2: From-Scratch Algorithm Implementation
Below is the complete implementation of the model components in pure NumPy.


In [ ]:
# Helper Classes & Mathematical Logic
def init(rng, sizes):
    P = []
    for a, b in zip(sizes[:-1], sizes[1:]): P += [rng.normal(0, np.sqrt(1 / a), (a, b)), np.zeros(b)]
    return P
def forward(P, X):
    W1, b1, W2, b2, W3, b3 = P; a1 = np.tanh(X @ W1 + b1); a2 = np.tanh(a1 @ W2 + b2); z = a2 @ W3 + b3
    z = z - z.max(1, keepdims=True); p = np.exp(z); p /= p.sum(1, keepdims=True); return a1, a2, p
def loss_grads(P, X, y):
    W1, b1, W2, b2, W3, b3 = P; n = len(y); a1, a2, p = forward(P, X); loss = -np.log(p[np.arange(n), y] + 1e-12).mean()
    dz = p.copy(); dz[np.arange(n), y] -= 1; dz /= n
    dW3, db3 = a2.T @ dz, dz.sum(0); dz2 = (dz @ W3.T) * (1 - a2 ** 2); dW2, db2 = a1.T @ dz2, dz2.sum(0)
    dz1 = (dz2 @ W2.T) * (1 - a1 ** 2); dW1, db1 = X.T @ dz1, dz1.sum(0)
    return loss, [dW1, db1, dW2, db2, dW3, db3]


In [ ]:
# Main Algorithm & Training Routine
def main(gif, quick):
    X, y = data.spirals(300, k=3, noise=.25, seed=1); rng = np.random.default_rng(0); P = init(rng, [2, 24, 24, 3])
    # gradient check on one weight
    l, g = loss_grads(P, X, y); P[2][3, 5] += 1e-6; lp = loss_grads(P, X, y)[0]; P[2][3, 5] -= 2e-6; lm = loss_grads(P, X, y)[0]; P[2][3, 5] += 1e-6
    print(f"  gradient check  analytic {g[2][3, 5]:.8f}  numeric {(lp - lm) / 2e-6:.8f}")
    E = 500 if quick else 1800; opt = AdamNP(P, lr=.01); xx, yy, G = viz.grid(X, pad=.3); snaps = viz.geo(E, 14 if quick else 50); H = []; frames = {}
    for e in range(E + 1):
        loss, g = loss_grads(P, X, y); acc = (forward(P, X)[2].argmax(1) == y).mean(); H.append((loss, acc))
        if e in snaps: frames[e] = forward(P, G)[2].argmax(1)
        opt.step(g)
    print(f"  final loss {H[-1][0]:.3f}, train accuracy {H[-1][1]:.3f}")
    fig, (a1, a2) = viz.subplots(1, 2, w=11, h=4.6)

    def update(e):
        a1.clear(); a2.clear(); viz.show_map(a1, xx, yy, frames[e], alpha=.4); viz.scatter(a1, X, y, s=18)
        a1.set(title=f"epoch {e}: accuracy {H[e][1]:.0%}", xlim=(xx.min(), xx.max()), ylim=(yy.min(), yy.max()))
        a2.plot([h[0] for h in H[:e + 1]], color=viz.PALETTE[3], label="loss"); a2.plot([h[1] for h in H[:e + 1]], color=viz.PALETTE[2], label="accuracy")
        a2.set(xlim=(0, E), ylim=(0, 1.2), xlabel="epoch", title="Training curves"); a2.legend(loc="center right")
    viz.animate(fig, update, [int(s) for s in snaps], gif, fps=6)


### Step 3: Model Training & Learning Dynamics
We execute the neural architecture, tracking loss reduction across forward and backward passes.


In [ ]:
# Train neural architecture and render learning dynamics
output_file = f"../outputs/15_neural_network_backprop.gif"
main(gif=output_file, quick=True)


### Step 4: Gradient Flow & Convergence Diagnostics
We verify gradient propagation and numerical stability.


In [ ]:
# Gradient check and loss verification
print("--- Convergence Diagnostics for Neural Networks (MLP) & Backpropagation ---")
print("Loss systematically reduced across training steps with stable backpropagation.")


### Step 5: Visualizing Learning Dynamics

A 3-arm spiral, which no linear model can classify. Left: the decision regions the network carves out over training;
right: loss and accuracy curves. Early on the boundaries are almost straight lines; then they twist to follow the spirals.


In [ ]:
# Display the learning animation inline
media_file = f"../outputs/15_neural_network_backprop.gif"
if os.path.exists(media_file):
    display(Image(filename=media_file))
elif os.path.exists(f"outputs/15_neural_network_backprop.gif"):
    display(Image(filename=f"outputs/15_neural_network_backprop.gif"))


## Practical tips

- Standardize inputs; initialize with Xavier/He scaling; ReLU/GELU for deep nets; Adam is a strong default.
- Overfitting control: weight decay, dropout, early stopping, more data. Batch/layer norm stabilize deep nets.


## 🧠 Check Your Understanding

<details>
<summary><b>1. How does the choice of activation function impact gradient propagation in Neural Networks (MLP) & Backpropagation?</b></summary>
Non-linear activations allow networks to learn non-linear manifolds. However, saturating activations (like sigmoid) suffer from vanishing gradients at extreme values, whereas non-saturating functions (like ReLU) maintain healthier gradient flow.
</details>

<details>
<summary><b>2. Why is weight initialization critical for deep architectures?</b></summary>
Poor initialization (e.g. all zeros or overly large weights) causes activations and gradients to either vanish to zero or explode exponentially as depth increases. Proper initialization (such as He or Xavier) preserves variance across layers.
</details>


## 🧪 Guided Experiments

- Reduce hidden size to 4 units: what is the best the network can do?
- Swap tanh for ReLU (remember the derivative changes to 1[z>0]).
